# Citation structure — ECHR precedent network + Austrian principle lifecycles

**Topic-agnostic** (Bucket-2 family): both analyses run on structured citation metadata that
`master_import.ipynb` captures for *any* keyword corpus from these sources — swap the topic,
re-run unchanged. No NLP, no labels, no uncertainty machinery needed.

- **Part A — ECHR precedent network.** Every judgment's `scl` field lists the Strasbourg
  case-law it cites. Parsing it yields: *which precedents structure this area of law*
  (in-degree), *how old the cited law is* (precedent age at citation), and *how concentrated*
  the precedent base is over time.
- **Part B — Austrian principle lifecycles.** Every Rechtssatz carries its full list of
  applying decisions with dates (`entscheidungstexte`) — the **application time-series of a
  legal principle**: birth, peak, decline, and whether it is still "living law".

Caveats stated with the numbers: `scl` coverage is partial (older/committee judgments often
lack it); RIS applications are known only via RIS's own linkage; the corpus is
keyword-matched, so these describe *this corpus's* citation structure, not the courts' full
docket.

## Part A — ECHR precedent network (from `scl`)

In [ ]:
import json
import re
from pathlib import Path
from collections import Counter, defaultdict

DATA_DIR, FIG_DIR, REPORT_DIR = Path("../data"), Path("../figures"), Path("../reports")
echr = json.loads((DATA_DIR / "echr_parental_alienation.json").read_text())

def ecli_year(r):
    m = re.match(r"ECLI:CE:ECHR:(\d{4})", r.get("ecli") or "")
    return int(m.group(1)) if m else None

_YEAR = re.compile(r"(19[5-9]\d|20[0-2]\d)")

def parse_scl(scl):
    """One citation per ';' fragment -> (case name, cited year|None)."""
    out = []
    for frag in (scl or "").split(";"):
        frag = frag.strip()
        if not frag:
            continue
        name = re.split(r",\s*(?:no\.|nos\.|\(dec\.\)|judgment|\d)", frag)[0].strip()
        name = re.sub(r"\s+", " ", name).rstrip(",")
        if len(name) < 5 or " v. " not in name and " v " not in name:
            continue
        years = _YEAR.findall(frag)
        cited_year = int(years[-1]) if years else None
        out.append((name, cited_year))
    return out

edges = []           # (citing_id, citing_year, cited_name, cited_year)
with_scl = 0
for r in echr:
    cites = parse_scl(r.get("scl"))
    if cites:
        with_scl += 1
        cy = ecli_year(r)
        for name, year in cites:
            edges.append((r.get("itemid"), cy, name, year))

print(f"cases with scl citations: {with_scl}/{len(echr)} "
      f"({with_scl/len(echr)*100:.0f}% -- coverage caveat) | citation edges: {len(edges)}")

cases with scl citations: 414/1116 (37% -- coverage caveat) | citation edges: 7293


In [ ]:
indeg = Counter(name for _, _, name, _ in edges)
n_citing = with_scl
print(f"distinct precedents cited: {len(indeg)}\n")
print("TOP 15 PRECEDENTS structuring this jurisprudence (in-degree):")
for name, n in indeg.most_common(15):
    print(f"  {n:4d}  ({n/n_citing*100:4.0f}% of citing cases)  {name}")

distinct precedents cited: 3346

TOP 15 PRECEDENTS structuring this jurisprudence (in-degree):
    75  (  18% of citing cases)  Hokkanen v. Finland
    56  (  14% of citing cases)  Kutzner v. Germany
    55  (  13% of citing cases)  Elsholz v. Germany [GC]
    54  (  13% of citing cases)  Ignaccolo-Zenide v. Romania
    52  (  13% of citing cases)  Keegan v. Ireland
    49  (  12% of citing cases)  Neulinger and Shuruk v. Switzerland [GC]
    43  (  10% of citing cases)  Scozzari and Giunta v. Italy [GC]
    43  (  10% of citing cases)  K. and T. v. Finland [GC]
    42  (  10% of citing cases)  Sommerfeld v. Germany [GC]
    42  (  10% of citing cases)  Johansen v. Norway
    38  (   9% of citing cases)  T.P. and K.M. v. the United Kingdom [GC]
    38  (   9% of citing cases)  W. v. the United Kingdom
    37  (   9% of citing cases)  Sahin v. Germany [GC]
    37  (   9% of citing cases)  X v. Latvia [GC]
    35  (   8% of citing cases)  Olsson v. Sweden (no. 1)


In [ ]:
ages = [cy - yr for _, cy, _, yr in edges if cy and yr and 0 <= cy - yr <= 70]
import numpy as np
ages = np.array(ages)
print(f"precedent age at citation (n={len(ages)} dated edges):")
print(f"  median {np.median(ages):.0f} years | mean {ages.mean():.1f} | p90 {np.percentile(ages, 90):.0f}")

# does the precedent base age or renew over time?
by_period = {}
for _, cy, _, yr in edges:
    if cy and yr and 0 <= cy - yr <= 70:
        period = "2000-2012" if cy <= 2012 else "2013-2025"
        by_period.setdefault(period, []).append(cy - yr)
print("\nmedian precedent age by citing period:")
for p, v in sorted(by_period.items()):
    print(f"  {p}: {np.median(v):.0f} years (n={len(v)})")

# concentration: share of all citations captured by the top-10 precedents, per period
print("\nprecedent concentration (share of citations to top-10 precedents of the period):")
for p in sorted(by_period):
    period_edges = [name for _, cy, name, _ in edges
                    if cy and ((cy <= 2012) == (p == "2000-2012"))]
    c = Counter(period_edges)
    top10 = sum(n for _, n in c.most_common(10))
    print(f"  {p}: {top10/len(period_edges)*100:.0f}%  ({len(period_edges)} citations, "
          f"{len(c)} distinct precedents)")

precedent age at citation (n=7270 dated edges):
  median 8 years | mean 10.0 | p90 21

median precedent age by citing period:
  2000-2012: 7 years (n=2133)
  2013-2025: 9 years (n=5137)

precedent concentration (share of citations to top-10 precedents of the period):
  2000-2012: 13%  (2149 citations, 1071 distinct precedents)
  2013-2025: 6%  (5144 citations, 2626 distinct precedents)


In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

top = indeg.most_common(15)[::-1]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5.5), gridspec_kw={"width_ratios": [3, 2]})
ax1.barh([n[:44] for n, _ in top], [c for _, c in top], color="#2c7fb8")
ax1.set_xlabel("citations received (in corpus)")
ax1.set_title("Top precedents — ECHR contact/family jurisprudence")
ax1.tick_params(labelsize=8)
ax2.hist(ages, bins=range(0, 45, 3), color="#7fcdbb", edgecolor="white")
ax2.axvline(np.median(ages), color="#d95f02", lw=2, label=f"median {np.median(ages):.0f}y")
ax2.set_xlabel("precedent age at citation (years)")
ax2.set_title("How old is the law being cited?")
ax2.legend()
fig.tight_layout()
out = FIG_DIR / "echr_precedent_network.png"
fig.savefig(out, dpi=130); plt.close(fig)
print("wrote", out.name)

wrote echr_precedent_network.png


## Part B — Austrian Rechtssatz lifecycles (from `entscheidungstexte`)

In [ ]:
ris = json.loads((DATA_DIR / "ris_parental_alienation.json").read_text())
rs_recs = [r for r in ris if r.get("dokumenttyp") == "Rechtssatz"]

lifecycles = {}      # rsnum -> sorted list of application years
for r in rs_recs:
    rsnum = (r.get("rechtssatznummern") or "").strip() or r.get("id")
    years = []
    for it in (r.get("entscheidungstexte") or []):
        d = (it.get("Entscheidungsdatum") or "")[:4]
        if d.isdigit():
            years.append(int(d))
    if years:
        lifecycles[rsnum] = sorted(years)

print(f"Rechtssaetze with dated applications: {len(lifecycles)}/{len(rs_recs)}")
print("\nRS lifecycle summary (top 10 by application count):")
print(f"{'RS':14s} {'n':>4s} {'first':>6s} {'last':>6s} {'span':>5s}  status")
for rsnum, ys in sorted(lifecycles.items(), key=lambda x: -len(x[1]))[:10]:
    span = ys[-1] - ys[0]
    status = "LIVING (applied in last 5y)" if ys[-1] >= 2021 else "dormant"
    print(f"{rsnum:14s} {len(ys):4d} {ys[0]:6d} {ys[-1]:6d} {span:4d}y  {status}")

living = sum(1 for ys in lifecycles.values() if ys[-1] >= 2021)
spans = [ys[-1] - ys[0] for ys in lifecycles.values()]
import numpy as np
print(f"\n'living law' share (applied since 2021): {living}/{len(lifecycles)} "
      f"= {living/len(lifecycles)*100:.0f}%")
print(f"principle active span: median {np.median(spans):.0f}y, max {max(spans)}y "
      f"-- distilled principles outlive individual decisions by decades")

Rechtssaetze with dated applications: 38/38

RS lifecycle summary (top 10 by application count):
RS                n  first   last  span  status
RS0007101       146   1990   2025   35y  LIVING (applied in last 5y)
RS0115719       115   2001   2025   24y  LIVING (applied in last 5y)
RS0049070       108   1953   2020   67y  dormant
RS0048633        94   1980   2025   45y  LIVING (applied in last 5y)
RS0006893        74   1961   2025   64y  LIVING (applied in last 5y)
RS0047955        47   1978   2025   47y  LIVING (applied in last 5y)
RS0048072        38   1979   2025   46y  LIVING (applied in last 5y)
RS0056290        37   1966   2021   55y  LIVING (applied in last 5y)
RS0074568        35   1992   2025   33y  LIVING (applied in last 5y)
RS0007310        33   1960   2025   65y  LIVING (applied in last 5y)

'living law' share (applied since 2021): 27/38 = 71%
principle active span: median 30y, max 70y -- distilled principles outlive individual decisions by decades


In [ ]:
top9 = sorted(lifecycles.items(), key=lambda x: -len(x[1]))[:9]
fig, axes = plt.subplots(3, 3, figsize=(12, 8), sharex=True)
for ax, (rsnum, ys) in zip(axes.flat, top9):
    c = Counter(ys)
    xs = list(range(min(ys), max(ys) + 1))
    ax.bar(xs, [c.get(x, 0) for x in xs], color="#2c7fb8", width=0.9)
    ax.set_title(f"{rsnum}  (n={len(ys)}, {ys[0]}-{ys[-1]})", fontsize=9)
    ax.tick_params(labelsize=7)
fig.suptitle("Lifecycles of Austrian legal principles — applications per year", y=1.0)
fig.tight_layout()
out = FIG_DIR / "ris_principle_lifecycles.png"
fig.savefig(out, dpi=130); plt.close(fig)
print("wrote", out.name)

wrote ris_principle_lifecycles.png


## Report

In [ ]:
lines = ["# Citation structure report (topic-agnostic Bucket-2 family)\n\n"]
lines.append(f"## ECHR precedent network\n"
             f"- coverage: {with_scl}/{len(echr)} cases carry `scl` citations "
             f"({with_scl/len(echr)*100:.0f}%); {len(edges)} citation edges, "
             f"{len(indeg)} distinct precedents.\n"
             f"- top precedents: " +
             "; ".join(f"**{n}** ({c})" for n, c in indeg.most_common(5)) + "\n"
             f"- precedent age at citation: median **{np.median(ages):.0f} years** "
             f"(p90 {np.percentile(ages, 90):.0f}) -- the Court builds on settled law.\n"
             f"- figure: `figures/echr_precedent_network.png`\n\n")
lines.append(f"## Austrian principle lifecycles\n"
             f"- {len(lifecycles)} Rechtssaetze with dated applications; median active span "
             f"**{np.median(spans):.0f} years** (max {max(spans)}).\n"
             f"- living law: **{living}/{len(lifecycles)}** principles applied since 2021.\n"
             f"- figure: `figures/ris_principle_lifecycles.png`\n\n")
lines.append("## Reusability note\n"
             "Both analyses consume only structured citation metadata captured by the "
             "generic importer -- they run **unchanged** on any topic corpus from these "
             "sources. Caveats: `scl` coverage is partial and biased toward judgments; RIS "
             "application lists come from RIS's own linkage; counts describe the "
             "keyword-matched corpus.\n")
out = REPORT_DIR / "citation_network_report.md"
out.write_text("".join(lines), encoding="utf-8")
print("wrote", out)

wrote ../reports/citation_network_report.md
